In [28]:
import pandas as pd

# 模擬三大法人每日買賣超資料（像 TEJ 抓下來的格式）
# 所有資料
chip_data = {
    "日期": ["2025-01-02", "2025-01-02", "2025-01-02",
             "2025-01-02", "2025-01-02", "2025-01-02",
             "2025-01-03", "2025-01-03", "2025-01-03",
             "2025-01-03", "2025-01-03", "2025-01-03",
             "2025-01-06", "2025-01-06", "2025-01-06",
             "2025-01-06", "2025-01-06", "2025-01-06"],
    "股票代號": ["2330", "2330", "2330", "2317", "2317", "2317",
                "2330", "2330", "2330", "2317", "2317", "2317",
                "2330", "2330", "2330", "2317", "2317", "2317"],
    "公司名稱": ["台積電", "台積電", "台積電", "鴻海", "鴻海", "鴻海",
                "台積電", "台積電", "台積電", "鴻海", "鴻海", "鴻海",
                "台積電", "台積電", "台積電", "鴻海", "鴻海", "鴻海"],
    "法人類別": ["外資", "投信", "自營商", "外資", "投信", "自營商",
                "外資", "投信", "自營商", "外資", "投信", "自營商",
                "外資", "投信", "自營商", "外資", "投信", "自營商"],
    "買賣超張數": [5000, 200, -150, -3000, 800, 200,
                  2000, -100, 300, 1500, 500, -400,
                  -8000, 600, -200, 2000, -300, 100]
}

df = pd.DataFrame(chip_data)
print(f"共 {len(df)} 筆資料")
df.head(6)

共 18 筆資料


,日期,股票代號,公司名稱,法人類別,買賣超張數
0,2025-01-02,2330,台積電,外資,5000
1,2025-01-02,2330,台積電,投信,200
2,2025-01-02,2330,台積電,自營商,-150
3,2025-01-02,2317,鴻海,外資,-3000
4,2025-01-02,2317,鴻海,投信,800
5,2025-01-02,2317,鴻海,自營商,200


In [29]:
# 【基本 pivot_table】
# 看兩支股票這個期間內的三大法人總買賣超
pivot_basic = pd.pivot_table(
    df,                        # 用哪個 DataFrame
    values="買賣超張數",        # 要計算哪個數字欄位
    index="股票代號",           # 列（rows）用什麼當分組
    columns="法人類別",         # 欄（columns）用什麼展開
    aggfunc="sum"              # 計算方式：加總
)

pivot_basic

法人類別,外資,投信,自營商
股票代號,,,
2317,500,1000,-100
2330,-1000,700,-50


In [30]:
# 兩支股票這個期間內的三大法人平均買賣超
pivot_mean=pd.pivot_table(
    df,
    values="買賣超張數",
    index="股票代號",
    columns="法人類別",
    aggfunc="mean"
).round(0)
pivot_mean

法人類別,外資,投信,自營商
股票代號,,,
2317,167.0,333.0,-33.0
2330,-333.0,233.0,-17.0


In [31]:
# 【多層 index】
#  每一天、每支股票，三種法人各買賣多少？

pivot_multi = pd.pivot_table(
    df,
    values="買賣超張數",
    index=["日期", "股票代號"],   # 注意：變成清單，兩個欄位
    columns="法人類別",
    aggfunc="sum"
)

pivot_multi

法人類別               外資   投信  自營商
日期         股票代號                
2025-01-02 2317 -3000  800  200
           2330  5000  200 -150
2025-01-03 2317  1500  500 -400
           2330  2000 -100  300
2025-01-06 2317  2000 -300  100
           2330 -8000  600 -200

In [32]:
#不用column
pivot_no_col = pd.pivot_table(
    df,
    values="買賣超張數",
    index="股票代號",
    aggfunc="sum"
)
pivot_no_col

,買賣超張數
股票代號,
2317,1400
2330,-350


In [33]:
#多層 column
pivot_multi_col = pd.pivot_table(
    df,
    values="買賣超張數",
    index="股票代號",
    columns=["法人類別", "日期"],    # 兩層欄：先法人再日期
    aggfunc="sum"
)
pivot_multi_col

法人類別         外資                               投信                        \
日期   2025-01-02 2025-01-03 2025-01-06 2025-01-02 2025-01-03 2025-01-06   
股票代號                                                                     
2317      -3000       1500       2000        800        500       -300   
2330       5000       2000      -8000        200       -100        600   

法人類別        自營商                        
日期   2025-01-02 2025-01-03 2025-01-06  
股票代號                                   
2317        200       -400        100  
2330       -150        300       -200

In [34]:
#將時間從字串轉換成日期格式
print(df["日期"].dtype) #轉換前的
df["日期"]=pd.to_datetime(df["日期"]) #轉換
print(df["日期"].dtype) #轉換後的

str
datetime64[us]


In [35]:
# dt 練習
print(df["日期"].dt.day_name())
#dt.year取出年份, month 取月,day,weekday取星期幾(星期一=0，星期日=6),day_name()=英文星期幾



0     Thursday
1     Thursday
2     Thursday
3     Thursday
4     Thursday
5     Thursday
6       Friday
7       Friday
8       Friday
9       Friday
10      Friday
11      Friday
12      Monday
13      Monday
14      Monday
15      Monday
16      Monday
17      Monday
Name: 日期, dtype: str


In [36]:
#在資料庫中新增日期格式的年月日和星期幾
df["年份"] = df["日期"].dt.year
df["月份"] = df["日期"].dt.month
df["星期"] = df["日期"].dt.day_name()
df.head(6)

,日期,股票代號,公司名稱,法人類別,買賣超張數,年份,月份,星期
0,2025-01-02,2330,台積電,外資,5000,2025,1,Thursday
1,2025-01-02,2330,台積電,投信,200,2025,1,Thursday
2,2025-01-02,2330,台積電,自營商,-150,2025,1,Thursday
3,2025-01-02,2317,鴻海,外資,-3000,2025,1,Thursday
4,2025-01-02,2317,鴻海,投信,800,2025,1,Thursday
5,2025-01-02,2317,鴻海,自營商,200,2025,1,Thursday


In [37]:
#實際練習，各法人各個月份的買賣超合計
pd.pivot_table(
    df,
    values="買賣超張數",
    index=["月份","日期","股票代號"],
    columns="法人類別",
    aggfunc="sum"
)


法人類別                  外資   投信  自營商
月份 日期         股票代號                
1  2025-01-02 2317 -3000  800  200
              2330  5000  200 -150
   2025-01-03 2317  1500  500 -400
              2330  2000 -100  300
   2025-01-06 2317  2000 -300  100
              2330 -8000  600 -200

In [38]:
df[df["日期"]=="2025-01-03"]

,日期,股票代號,公司名稱,法人類別,買賣超張數,年份,月份,星期
6,2025-01-03,2330,台積電,外資,2000,2025,1,Friday
7,2025-01-03,2330,台積電,投信,-100,2025,1,Friday
8,2025-01-03,2330,台積電,自營商,300,2025,1,Friday
9,2025-01-03,2317,鴻海,外資,1500,2025,1,Friday
10,2025-01-03,2317,鴻海,投信,500,2025,1,Friday
11,2025-01-03,2317,鴻海,自營商,-400,2025,1,Friday


In [39]:
# 2025-01-03 以後（含當天）
df[df["日期"] >= "2025-01-03"]



,日期,股票代號,公司名稱,法人類別,買賣超張數,年份,月份,星期
6,2025-01-03,2330,台積電,外資,2000,2025,1,Friday
7,2025-01-03,2330,台積電,投信,-100,2025,1,Friday
8,2025-01-03,2330,台積電,自營商,300,2025,1,Friday
9,2025-01-03,2317,鴻海,外資,1500,2025,1,Friday
10,2025-01-03,2317,鴻海,投信,500,2025,1,Friday
11,2025-01-03,2317,鴻海,自營商,-400,2025,1,Friday
12,2025-01-06,2330,台積電,外資,-8000,2025,1,Monday
13,2025-01-06,2330,台積電,投信,600,2025,1,Monday
14,2025-01-06,2330,台積電,自營商,-200,2025,1,Monday
15,2025-01-06,2317,鴻海,外資,2000,2025,1,Monday


In [40]:
# 2025-01-02 到 2025-01-03（用 AND）
df[(df["日期"] >= "2025-01-02") & (df["日期"] <= "2025-01-03")]

,日期,股票代號,公司名稱,法人類別,買賣超張數,年份,月份,星期
0,2025-01-02,2330,台積電,外資,5000,2025,1,Thursday
1,2025-01-02,2330,台積電,投信,200,2025,1,Thursday
2,2025-01-02,2330,台積電,自營商,-150,2025,1,Thursday
3,2025-01-02,2317,鴻海,外資,-3000,2025,1,Thursday
4,2025-01-02,2317,鴻海,投信,800,2025,1,Thursday
5,2025-01-02,2317,鴻海,自營商,200,2025,1,Thursday
6,2025-01-03,2330,台積電,外資,2000,2025,1,Friday
7,2025-01-03,2330,台積電,投信,-100,2025,1,Friday
8,2025-01-03,2330,台積電,自營商,300,2025,1,Friday
9,2025-01-03,2317,鴻海,外資,1500,2025,1,Friday


In [41]:
#練習用 .between() 篩選範圍
df[df["日期"].between("2025-01-02", "2025-01-03")] 

#跟上面的df[(df["日期"] >= "2025-01-02") & (df["日期"] <= "2025-01-03")]一樣都是取1月2到1月3號間得資料


,日期,股票代號,公司名稱,法人類別,買賣超張數,年份,月份,星期
0,2025-01-02,2330,台積電,外資,5000,2025,1,Thursday
1,2025-01-02,2330,台積電,投信,200,2025,1,Thursday
2,2025-01-02,2330,台積電,自營商,-150,2025,1,Thursday
3,2025-01-02,2317,鴻海,外資,-3000,2025,1,Thursday
4,2025-01-02,2317,鴻海,投信,800,2025,1,Thursday
5,2025-01-02,2317,鴻海,自營商,200,2025,1,Thursday
6,2025-01-03,2330,台積電,外資,2000,2025,1,Friday
7,2025-01-03,2330,台積電,投信,-100,2025,1,Friday
8,2025-01-03,2330,台積電,自營商,300,2025,1,Friday
9,2025-01-03,2317,鴻海,外資,1500,2025,1,Friday


In [42]:
#按日期排序
df=df.sort_values("日期")
df.head(6)

,日期,股票代號,公司名稱,法人類別,買賣超張數,年份,月份,星期
0,2025-01-02,2330,台積電,外資,5000,2025,1,Thursday
1,2025-01-02,2330,台積電,投信,200,2025,1,Thursday
2,2025-01-02,2330,台積電,自營商,-150,2025,1,Thursday
3,2025-01-02,2317,鴻海,外資,-3000,2025,1,Thursday
4,2025-01-02,2317,鴻海,投信,800,2025,1,Thursday
5,2025-01-02,2317,鴻海,自營商,200,2025,1,Thursday


In [43]:
#把日期設成索引
df=df.set_index("日期")
df.head(6)

,股票代號,公司名稱,法人類別,買賣超張數,年份,月份,星期
日期,,,,,,,
2025-01-02,2330,台積電,外資,5000,2025,1,Thursday
2025-01-02,2330,台積電,投信,200,2025,1,Thursday
2025-01-02,2330,台積電,自營商,-150,2025,1,Thursday
2025-01-02,2317,鴻海,外資,-3000,2025,1,Thursday
2025-01-02,2317,鴻海,投信,800,2025,1,Thursday
2025-01-02,2317,鴻海,自營商,200,2025,1,Thursday


In [ ]:
# 取特定日期
df.loc["2025-01-03"]

,股票代號,公司名稱,法人類別,買賣超張數,年份,月份,星期
日期,,,,,,,
2025-01-03,2317,鴻海,自營商,-400,2025,1,Friday
2025-01-03,2317,鴻海,投信,500,2025,1,Friday
2025-01-03,2317,鴻海,外資,1500,2025,1,Friday
2025-01-03,2330,台積電,自營商,300,2025,1,Friday
2025-01-03,2330,台積電,投信,-100,2025,1,Friday
2025-01-03,2330,台積電,外資,2000,2025,1,Friday


In [ ]:
# 取日期範圍（loc切片語法含頭尾，但一般的python切片語法不包含尾巴) 
df.loc["2025-01-02":"2025-01-03"]

,股票代號,公司名稱,法人類別,買賣超張數,年份,月份,星期
日期,,,,,,,
2025-01-02,2330,台積電,外資,5000,2025,1,Thursday
2025-01-02,2330,台積電,投信,200,2025,1,Thursday
2025-01-02,2330,台積電,自營商,-150,2025,1,Thursday
2025-01-02,2317,鴻海,外資,-3000,2025,1,Thursday
2025-01-02,2317,鴻海,投信,800,2025,1,Thursday
2025-01-02,2317,鴻海,自營商,200,2025,1,Thursday
2025-01-03,2317,鴻海,自營商,-400,2025,1,Friday
2025-01-03,2317,鴻海,投信,500,2025,1,Friday
2025-01-03,2317,鴻海,外資,1500,2025,1,Friday


In [46]:
#先把日期從索引單位轉回普通欄位
df = df.reset_index()